In [1]:
import time
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model_cpu = AutoModel.from_pretrained("bert-base-uncased").to("cpu")
model_gpu = AutoModel.from_pretrained("bert-base-uncased").to("cuda")

model_cpu.eval()
model_gpu.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [2]:
text = "Benchmarking inference speed helps decide deployment tradeoffs." * 5  # longer text
inputs_cpu = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
inputs_gpu = {k: v.to("cuda") for k, v in inputs_cpu.items()}

# CPU timing
with torch.no_grad():
    start = time.time()
    for _ in range(10):
        _ = model_cpu(**inputs_cpu)
    cpu_time = (time.time() - start) / 10

# GPU timing
with torch.no_grad():
    torch.cuda.synchronize()  # wait for all queued GPU ops to finish before timing
    start = time.time()
    for _ in range(10):
        _ = model_gpu(**inputs_gpu)
    torch.cuda.synchronize()  # wait again before stopping the clock
    gpu_time = (time.time() - start) / 10

print(f"CPU avg time: {cpu_time*1000:.2f} ms")
print(f"GPU avg time: {gpu_time*1000:.2f} ms")
print(f"Speedup: {cpu_time/gpu_time:.1f}x")

CPU avg time: 79.82 ms
GPU avg time: 28.93 ms
Speedup: 2.8x


In [3]:
texts = [text] * 16  # simulate 16 requests arriving together
inputs_cpu_batch = tokenizer(texts, return_tensors="pt", truncation=True, max_length=128, padding=True)
inputs_gpu_batch = {k: v.to("cuda") for k, v in inputs_cpu_batch.items()}

with torch.no_grad():
    start = time.time()
    _ = model_cpu(**inputs_cpu_batch)
    cpu_batch_time = time.time() - start

with torch.no_grad():
    torch.cuda.synchronize()
    start = time.time()
    _ = model_gpu(**inputs_gpu_batch)
    torch.cuda.synchronize()
    gpu_batch_time = time.time() - start

print(f"CPU batch(16) time: {cpu_batch_time*1000:.2f} ms")
print(f"GPU batch(16) time: {gpu_batch_time*1000:.2f} ms")
print(f"Speedup: {cpu_batch_time/gpu_batch_time:.1f}x")

CPU batch(16) time: 669.43 ms
GPU batch(16) time: 38.74 ms
Speedup: 17.3x


In [4]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

with torch.no_grad():
    _ = model_gpu(**inputs_gpu_batch)

allocated = torch.cuda.memory_allocated() / 1024**2  # convert bytes to MB
peak = torch.cuda.max_memory_allocated() / 1024**2

print(f"Currently allocated: {allocated:.1f} MB")
print(f"Peak allocated during forward pass: {peak:.1f} MB")

Currently allocated: 429.6 MB
Peak allocated during forward pass: 459.0 MB


In [5]:
total = torch.cuda.get_device_properties(0).total_memory / 1024**2
reserved = torch.cuda.memory_reserved() / 1024**2
free = total - reserved

print(f"Total VRAM: {total:.0f} MB")
print(f"Reserved by PyTorch: {reserved:.0f} MB")
print(f"Roughly free: {free:.0f} MB")

Total VRAM: 6140 MB
Reserved by PyTorch: 516 MB
Roughly free: 5624 MB
